# Deploy the Customer H2O Model Online

Create or reuse the configured endpoint, deploy the validated customer model at zero traffic, inspect logs, invoke the deployment with golden input, and optionally promote traffic.

**Source:** Adapted from this repository's H2O endpoint notebook and the Azure ML simple managed endpoint example.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from azure.ai.ml import MLClient
from azure.ai.ml.constants import ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def workshop_path(name: str) -> Path:
    value = Path(os.environ[name])
    return value if value.is_absolute() else WORKSHOP_ROOT / value

MODEL_PATH = workshop_path("H2O_CUSTOMER_MODEL_PATH")
INPUT_PATH = workshop_path("H2O_CUSTOMER_INPUT_PATH")
EXPECTED_PATH = workshop_path("H2O_CUSTOMER_EXPECTED_PATH")
BUNDLE_DIR = MODEL_PATH.resolve().parent
manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]
ENVIRONMENT_NAME = os.environ["H2O_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["H2O_ENVIRONMENT_VERSION"]
ENDPOINT_NAME = os.environ["H2O_ENDPOINT_NAME"]
DEPLOYMENT_NAME = os.environ["H2O_DEPLOYMENT_NAME"]
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
DEPLOY = os.getenv("DEPLOY_H2O_ENDPOINT", "false").lower() in {"1", "true", "yes"}
PROMOTE = os.getenv("PROMOTE_H2O_TRAFFIC", "false").lower() in {"1", "true", "yes"}

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[ManagedIdentityConfiguration(resource_id=IDENTITY_ID)],
    )
endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    auth_mode="aad_token",
    identity=identity,
    public_network_access=os.getenv("AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS", "disabled"),
    description="Customer H2O binary-model endpoint",
    tags={"workshop": "azureml-h2o", "model": f"{MODEL_NAME}:{MODEL_VERSION}"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}:{MODEL_VERSION}",
    environment=f"azureml:{ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}",
    code_configuration=CodeConfiguration(code=str(WORKSHOP_ROOT / "src/h2o/online"), scoring_script="score.py"),
    instance_type=os.environ["AZUREML_ONLINE_INSTANCE_TYPE"],
    instance_count=1,
    app_insights_enabled=True,
    environment_variables={"WORKER_COUNT": "1", "H2O_NTHREADS": os.environ["H2O_NTHREADS"], "H2O_MAX_MEM_SIZE": os.environ["H2O_MAX_MEM_SIZE"]},
)

if DEPLOY:
    ml_client.models.get(MODEL_NAME, MODEL_VERSION)
    ml_client.environments.get(ENVIRONMENT_NAME, ENVIRONMENT_VERSION)
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
    except ResourceNotFoundError:
        endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
    deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")
    logs = ml_client.online_deployments.get_logs(DEPLOYMENT_NAME, ENDPOINT_NAME, 100, container_type="inference-server")
    print("\n".join(logs.splitlines()[-25:]))

    golden_input = pd.read_csv(INPUT_PATH)
    golden_expected = pd.read_csv(EXPECTED_PATH)
    request = {"input_data": {"columns": manifest["features"], "data": golden_input[manifest["features"]].values.tolist()}}
    request_path = WORKSHOP_ROOT / "outputs/customer_h2o_request.json"
    request_path.write_text(json.dumps(request, indent=2), encoding="utf-8")
    response = json.loads(ml_client.online_endpoints.invoke(endpoint_name=ENDPOINT_NAME, deployment_name=DEPLOYMENT_NAME, request_file=str(request_path)))
    np.testing.assert_allclose(golden_expected["predict"], response["predictions"], rtol=1e-6, atol=1e-6)
    print(f"Cloud parity passed for {len(response['predictions'])} rows.")

    if PROMOTE:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint.traffic = {DEPLOYMENT_NAME: 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint).result()
        print(f"Traffic promoted to {DEPLOYMENT_NAME}")
    else:
        print("Traffic remains unchanged. Set PROMOTE_H2O_TRAFFIC=true to promote.")
else:
    print(f"Prepared endpoint/deployment: {ENDPOINT_NAME}/{DEPLOYMENT_NAME}")
    print("Deployment disabled. Set DEPLOY_H2O_ENDPOINT=true in workshop/.env.")

## Expected Result

The deployment reaches `Succeeded`, logs show initialization, direct invocation matches the customer's golden output, and traffic changes only when enabled.

Next: `05_submit_scoring_pipeline.ipynb`.